In [0]:
-- Temporary query to analyse messages
CREATE OR REPLACE VIEW sac.support.ai_analyse_message AS
WITH messages_raw AS (
	SELECT
		c.session_id,
		msg.speaker,
		CAST(msg.timestamp AS TIMESTAMP) AS timestamp,
		msg.message,
		c.ingestion_time
	FROM
		sac.support.chat_bronze c
			LEFT JOIN sac.support.message m
				ON c.session_id = m.session_id
		LATERAL VIEW EXPLODE(from_json(c.messages, 'ARRAY<MAP<STRING,STRING>>')) AS msg
	WHERE
		m.message IS NULL
),
opinion AS (
	SELECT
		messages_raw.session_id,
		messages_raw.speaker,
		messages_raw.timestamp,
		ai_query(
			"databricks-gemma-3-12b",
			request =>
				concat(
					"Du bist ein Internet Service Provider. Gebe basierend auf einem Textabschnitt als Ergebnis ein JSON Array aus, das eine Zusammenfassung, eine Klassifikation und ein Positiv, Negativ oder Neutral Sentiment über das Thema enthält. Klassifiziert muss einer der folgenden Anworten sein: CONNECTION ISSUES, SLOW SPEED, SERVICE, PRICE, OTHER. Du kannst keine Klassifikations Kategorie halluzinieren.

Beispiel:

DOCUMENT
Mein Router startet sich alle 15 Minuten von selbst neu. Das ist super nervig. (Gemessene Geschwindigkeit: 180 Mbps, Issue: packet_loss).

RESULT
[
{'Classification': 'CONNECTION ISSUES','Comment': 'Router startet ständig neu','Sentiment': 'Negativ'}
]

DOCUMENT\n",
					messages_raw.message,
					'\n\nRESULT\n'
				),
			responseFormat =>
				'{
                "type": "json_schema",
                "json_schema": {
                    "name": "opinion_mining_schema",
                    "schema": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "classification": { "type": "string" },
                                "comment": { "type": "string" },
                                "sentiment": { "type": "string" }
                            }
                        }
                    },
                    "strict": true
                }
            }'
		) AS extracted_opinions
	FROM
		messages_raw
)
SELECT
	m.session_id,
	m.speaker,
	m.timestamp,
	m.message,
	m.ingestion_time,
	op.classification,
	op.comment,
	op.sentiment
FROM
	messages_raw m
		LEFT JOIN opinion o
			ON m.session_id = o.session_id
			AND m.timestamp = o.timestamp
	LATERAL VIEW OUTER EXPLODE(from_json(o.extracted_opinions, 'ARRAY<MAP<STRING,STRING>>')) AS op